## System One: uma chamada, quatro decisões

Kahneman separa dois modos de pensar: o **System One**, rápido e
associativo, e o **System Two**, lento e deliberado. Até aqui o modelo
foi System Two: prompt montado à mão, exemplos, laço, uma decisão por
vez.

Esta demo começa pelo outro lado. O Jev expõe o **System One** por
HTTP: numa única chamada você manda um estado (o texto da reclamação)
e uma lista de perguntas **tipadas** — `choice`, `score`, `noul` — e
recebe de volta os valores já estruturados, com confiança e
probabilidades.

Sem prompt de sistema. Sem exemplos. Sem parse de texto.

In [1]:
# Setup: dados, credenciais e o helper do Jev (jev.py)
import json, os, pathlib, sys

sys.path.append(str(pathlib.Path.cwd().parent))

import pandas as pd
import jev

# O modelo vem do .env (JEV_MODEL); sem ele vale o default do jev.py.
MODELO = os.environ.get("JEV_MODEL") or jev._DEFAULT_MODEL
print(f"Modelo: {MODELO}")

df = pd.read_csv('../data/reclamacoes.csv')
df.head(3)

Modelo: jev-1.13-free


,texto,categoria
0,O rastreador do pedido não atualiza há duas se...,entrega
1,Cobraram o frete duas vezes na minha fatura do...,cobranca
2,O produto chegou com a embalagem amassada e se...,produto


In [2]:
# As quatro perguntas são fixadas uma vez, fora do laço. Repare no que
# NÃO existe: instrução de formato, lista de exemplos, "responda em
# JSON". O tipo da pergunta é o formato.
PERGUNTAS = {
    "categoria": jev.Choice(
        instructions="Classifique a reclamação em uma das categorias.",
        criteria={
            "entrega": "problemas de envio, prazo, rastreio ou extravio do pedido",
            "cobranca": "cobrança indevida, valor errado, estorno ou fatura",
            "produto": "defeito, avaria ou item diferente do anunciado",
            "atendimento": "dificuldade no contato com o suporte e no retorno",
            "app": "falha, erro ou travamento no aplicativo",
        },
    ),
    # Irritação como Choice, não Score: devolve a chave escolhida e se
    # repete entre execuções. O Score aparece no "Experimente", com o
    # comportamento real do upstream registrado no output.
    "irritacao": jev.Choice(
        instructions="Qual é o tom emocional da reclamação?",
        criteria={
            "calmo": "calmo, só fatos",
            "irritado": "irritado mas civilizado",
            "muito_irritado": "muito irritado, linguagem forte",
        },
    ),
    "urgente": jev.Noul(
        "A mensagem exige atenção imediata ou o cliente menciona prazo/tempo crítico"
    ),
    "dados_faltando": jev.Noul(
        "Faltam informações essenciais para resolver (nº do pedido, data, comprovante)"
    ),
}

# O que vai no corpo HTTP: cada pergunta vira protocolo, não texto.
print(json.dumps({nome: p.wire() for nome, p in PERGUNTAS.items()}, ensure_ascii=False, indent=2)[:1000])

{
  "categoria": {
    "type": "choice",
    "instructions": "Classifique a reclamação em uma das categorias.",
    "criteria": {
      "entrega": "problemas de envio, prazo, rastreio ou extravio do pedido",
      "cobranca": "cobrança indevida, valor errado, estorno ou fatura",
      "produto": "defeito, avaria ou item diferente do anunciado",
      "atendimento": "dificuldade no contato com o suporte e no retorno",
      "app": "falha, erro ou travamento no aplicativo"
    }
  },
  "irritacao": {
    "type": "choice",
    "instructions": "Qual é o tom emocional da reclamação?",
    "criteria": {
      "calmo": "calmo, só fatos",
      "irritado": "irritado mas civilizado",
      "muito_irritado": "muito irritado, linguagem forte"
    }
  },
  "urgente": {
    "type": "noul",
    "instructions": "A mensagem exige atenção imediata ou o cliente menciona prazo/tempo crítico"
  },
  "dados_faltando": {
    "type": "noul",
    "instructions": "Faltam informações essenciais para resolver (n

In [3]:
# Uma chamada por linha: 18 chamadas no total. O estado é o texto; as
# perguntas são as mesmas. O que muda a cada chamada é só a reclamação.
linhas = []
for i, linha in df.iterrows():
    respostas = jev.system_one(linha["texto"], PERGUNTAS)
    uso = jev.last_usage() or {}
    categoria = respostas["categoria"]
    linhas.append({
        "previsto": categoria["valor"],
        "confianca": categoria.get("confianca"),
        "irritacao": respostas["irritacao"]["valor"],
        "urgente": round(respostas["urgente"]["valor"], 2),
        "dados_faltando": round(respostas["dados_faltando"]["valor"], 2),
        "input_tokens": uso.get("input_tokens"),
        "output_tokens": uso.get("output_tokens"),
        "cost": uso.get("cost"),
    })
    marca = "OK   " if categoria["valor"] == linha["categoria"] else "ERROU"
    print(f"[{marca}] {i:2d} {linha['texto'][:48]:<48} -> {categoria['valor']:<11} conf={categoria.get('confianca')}")

res = pd.concat([df, pd.DataFrame(linhas)], axis=1)

# Guardamos o uso da triagem: os experimentos abaixo continuam somando.
USO_TRIAGEM = jev.usage_total()
print(f"\nTriagem (18 chamadas): {USO_TRIAGEM['input_tokens']} tokens in / {USO_TRIAGEM['output_tokens']} out")
print(f"Custo devolvido pelo gateway: US$ {USO_TRIAGEM['cost']:.6f} (billing: {(jev.last_usage() or {}).get('billing_source')})")

[OK   ]  0 O rastreador do pedido não atualiza há duas sema -> entrega     conf=1


[OK   ]  1 Cobraram o frete duas vezes na minha fatura do c -> cobranca    conf=1


[OK   ]  2 O produto chegou com a embalagem amassada e sem  -> produto     conf=1


[OK   ]  3 O atendente desligou o telefone na minha cara de -> atendimento conf=1


[OK   ]  4 O aplicativo trava na tela de pagamento e não de -> app         conf=1


...

[OK   ] 13 Três e-mails e dois telefones depois ninguém res -> atendimento conf=1


[OK   ] 14 Não consigo cadastrar o cartão novo, o app dá er -> app         conf=1


[ERROU] 15 O atendente confirmou o reembolso do produto que -> cobranca    conf=0.97


[ERROU] 16 O desconto do cupom do app sumiu no checkout e a -> app         conf=0.52


[OK   ] 17 A segunda parcela do pedido feito pelo app saiu  -> cobranca    conf=1

Triagem (18 chamadas): 10204 tokens in / 2539 out
Custo devolvido pelo gateway: US$ 0.000000 (billing: free)


In [4]:
# Placar contra o gabarito do CSV, no mesmo formato do demo-1.
acertos = (res["previsto"] == res["categoria"]).sum()
print(f"Placar da categoria: {acertos}/{len(res)} ({acertos/len(res):.0%})\n")

for _, linha in res[res["previsto"] != res["categoria"]].iterrows():
    print(f"[ERROU] esperado: {linha['categoria']:<11} | veio: {linha['previsto']:<11} {linha['texto'][:50]}")

# Onde o modelo hesita? Confiança arredondada para o crosstab não abrir
# uma coluna por centésimo.
pd.crosstab(res["categoria"], res["confianca"].round(2))

Placar da categoria: 16/18 (89%)

[ERROU] esperado: entrega     | veio: cobranca    O atendente confirmou o reembolso do produto que n
[ERROU] esperado: cobranca    | veio: app         O desconto do cupom do app sumiu no checkout e a c


confianca,0.52,0.97,1.00
categoria,,,
app,0,0,3
atendimento,0,0,3
cobranca,1,0,4
entrega,0,1,3
produto,0,0,3


In [5]:
# A política da triagem: confiança alta resolve sozinha, o resto vai
# para humano. O número 0.9 não veio do modelo; é decisão de operação.
THRESHOLD = 0.9

automaticas = res[res["confianca"] >= THRESHOLD]
revisao = res[res["confianca"] < THRESHOLD]

print(f"Threshold {THRESHOLD}: {len(automaticas)} automáticas, {len(revisao)} para revisão humana")
print(f"Taxa auto-decidable: {len(automaticas)/len(res):.0%}")
if len(automaticas):
    acerto_auto = (automaticas["previsto"] == automaticas["categoria"]).mean()
    print(f"Acerto entre as automáticas: {acerto_auto:.0%}")

# Confiança alta não é garantia de acerto - por isso a política precisa
# ser medida, não presumida. Aqui o gate acertou a mão: a única linha
# abaixo do threshold é também uma das que erraram. A outra passou
# como automática e passou errada.
for _, linha in revisao.iterrows():
    print(f"  revisar: {linha['texto'][:48]:<48} previsto={linha['previsto']:<11} conf={linha['confianca']}")

Threshold 0.9: 17 automáticas, 1 para revisão humana
Taxa auto-decidable: 94%
Acerto entre as automáticas: 94%
  revisar: O desconto do cupom do app sumiu no checkout e a previsto=app         conf=0.52


## Experimente

As decisões acima são política nossa. As quatro intervenções abaixo
mostram onde a política mora — e o que sobra para o modelo.

In [6]:
# (a) O threshold é política, não modelo: a confiança gravada é a mesma,
# só a régua muda. Subir aperta a fila de revisão; baixar afrouxa.
# A confiança chega em degraus, não numa curva: mexer no threshold só
# muda a fila quando ele cruza um degrau (0.9 e 0.7 dão o mesmo número).
for limite in (0.95, 0.9, 0.7, 0.6, 0.5):
    auto = int((res["confianca"] >= limite).sum())
    print(f"threshold {limite:.2f}: {auto:2d} automáticas | {len(res)-auto:2d} para revisão humana ({auto/len(res):.0%} auto)")

threshold 0.95: 17 automáticas |  1 para revisão humana (94% auto)
threshold 0.90: 17 automáticas |  1 para revisão humana (94% auto)
threshold 0.70: 17 automáticas |  1 para revisão humana (94% auto)
threshold 0.60: 17 automáticas |  1 para revisão humana (94% auto)
threshold 0.50: 18 automáticas |  0 para revisão humana (100% auto)


In [7]:
# (b) As mesmas perguntas com as instruções em inglês. Os nomes dos
# critérios são a taxonomia: não traduzimos, senão mudaríamos duas
# coisas de uma vez e a comparação perderia o sentido.
PERGUNTAS_EN = {
    "categoria": jev.Choice(
        instructions="Classify the complaint into one of the categories.",
        criteria={
            "entrega": "shipping, delivery deadline, tracking or lost order",
            "cobranca": "wrong charge, incorrect amount, refund or invoice",
            "produto": "defect, damage or item different from the listing",
            "atendimento": "trouble reaching support or getting a reply",
            "app": "app failure, error or crash",
        },
    ),
    "irritacao": jev.Choice(
        instructions="What is the emotional tone of the complaint?",
        criteria={
            "calmo": "calm, just the facts",
            "irritado": "annoyed but civil",
            "muito_irritado": "very angry, strong language",
        },
    ),
    "urgente": jev.Noul(
        "Does the message require immediate attention, or does the customer mention a critical deadline?"
    ),
    "dados_faltando": jev.Noul(
        "Is essential information missing to solve it (order number, date, receipt)?"
    ),
}

N = 5
previstos_en = [jev.system_one(df.iloc[i]["texto"], PERGUNTAS_EN)["categoria"]["valor"] for i in range(N)]
pt = res.iloc[:N].reset_index(drop=True)

placar_pt = (pt["previsto"] == pt["categoria"]).mean()
placar_en = (pd.Series(previstos_en) == pt["categoria"]).mean()
print(f"Primeiras {N} linhas - instruções em pt: {placar_pt:.0%} | em inglês: {placar_en:.0%}\n")
for i in range(N):
    print(f"  {i}: esperado={pt.loc[i, 'categoria']:<11} pt={pt.loc[i, 'previsto']:<11} en={previstos_en[i]}")
# As cinco primeiras são as fáceis (na triagem, todas com confiança 1.0):
# empate aqui é sanity check do protocolo, não benchmark de idioma.

Primeiras 5 linhas - instruções em pt: 100% | em inglês: 100%

  0: esperado=entrega     pt=entrega     en=entrega
  1: esperado=cobranca    pt=cobranca    en=cobranca
  2: esperado=produto     pt=produto     en=produto
  3: esperado=atendimento pt=atendimento en=atendimento
  4: esperado=app         pt=app         en=app


(c) A pergunta de irritação no formato `score` — o caminho que a demo
não usa. O comportamento do upstream varia de um dia para o outro;
rodamos e registramos o resultado real abaixo (nada de mock).

In [8]:
# (c) Score no lugar do Choice: uma chamada, várias notas. Se o upstream
# recusar, o erro real fica no output - é a jagged edge documentada.
try:
    score = jev.system_one(
        df.iloc[0]["texto"],
        {"irritacao_score": jev.Score(
            instructions="Pontue o tom emocional da reclamação.",
            criteria=["calmo", "irritado mas civilizado", "muito irritado, linguagem forte"],
        )},
    )
    print("Score respondeu:", score)
    # Para a política da triagem, o Choice entrega a chave escolhida; o
    # Score entrega uma nota e probabilidades indexadas por posição -
    # menos direto de consumir, ainda que responda.
except RuntimeError as exc:
    print("Score recusado pelo upstream:", exc)

Score respondeu: {'irritacao_score': {'tipo': 'score', 'valor': 0.99, 'confianca': 0.98, 'probabilidades': {'0': 0.01, '1': 0.99, '2': 0}}}


In [9]:
# (d) O modelo é peça substituível: JEV_MODEL no .env manda. O free
# devolve cost 0 e é tempo limitado; o jev-1.13 é o fallback pago a
# US$0.042 por milhão de tokens de input (output grátis): os ~13k
# tokens de input desta demo custam ~US$0.0006 no pago.
total = jev.usage_total()
extra_in = total["input_tokens"] - USO_TRIAGEM["input_tokens"]
extra_out = total["output_tokens"] - USO_TRIAGEM["output_tokens"]
print(f"Modelo em uso: {MODELO}")
print(f"Triagem (18 chamadas): {USO_TRIAGEM['input_tokens']:>6} in / {USO_TRIAGEM['output_tokens']:>5} out")
print(f"Experimentos:          {extra_in:>6} in / {extra_out:>5} out")
print(f"Processo inteiro:      {total['input_tokens']:>6} in / {total['output_tokens']:>5} out | custo US$ {total['cost']:.6f}")
# Para trocar de modelo: JEV_MODEL=jev-1.13 no .env e rodar de novo.

Modelo em uso: jev-1.13-free
Triagem (18 chamadas):  10204 in /  2539 out
Experimentos:            2991 in /   727 out
Processo inteiro:       13195 in /  3266 out | custo US$ 0.000000


## A lição

O modelo não decide o que fazer com a própria confiança: ele **mede**
(`0.8` é um número, não uma política). Quem decide é o código — o
threshold, o tamanho da fila de revisão, o que vai para humano. Trocar
`0.9` por `0.7` não muda uma vírgula da chamada HTTP; muda o processo.

Compare com o few-shot do demo-1: lá o placar da suíte chegou a 91%
depois de exemplos e regra no prompt. Aqui, sem prompt nenhum, o
número vem com a confiança acoplada — e é isso que permite calibrar.
A pergunta deixa de ser "o modelo acertou?" e passa a ser "quanto
custo aceito por quantos erros?".

O preço do juiz aparece no fim: `usage_total()` com o `cost` real
devolvido pelo gateway, sem conta na mão.